In [61]:
# you will be prompted with a window asking to grant permissions
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [62]:
# fill in the path in your Google Drive in the string below. Note: do not escape slashes or spaces
import os
datadir = "/content/drive/MyDrive/ECE_494_assignments/assignment4"
if not os.path.exists(datadir):
  !ln -s "/content/drive/MyDrive/ECE_494_assignments/assignment4" $datadir # TODO: Fill your Assignment 4 path
os.chdir(datadir)
!pwd

/content/drive/MyDrive/ECE_494_assignments/assignment4


In [63]:
# NOTE: Loading data from Google Drive is VERY SLOW. Therefore, we download the data to a local storage
# specified by VOC_PATH.
# This means the data would have be re-downloaded everytime.
# Change this path if you are running this script locally
VOC_PATH="sample_data/VOC_DATA"

In [64]:
!chmod u+x ./download_data.sh
!sed -i 's/\r//g' ./download_data.sh
!cat ./download_data.sh
!bash ./download_data.sh $VOC_PATH

In [65]:
import os
import random

import cv2
import numpy as np

import torch
from torch.utils.data import DataLoader

from src.resnet_yolo import resnet50
import yolo_loss
from src.dataset import VocDetectorDataset
from src.eval_voc import evaluate
from src.predict import predict_image
from src.config import VOC_CLASSES, COLORS
from kaggle_submission import output_submission_csv

import matplotlib.pyplot as plt
import collections

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Initialization

In [66]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [67]:
# YOLO network hyperparameters
B = 2  # number of bounding box predictions per cell
S = 14  # width/height of network output grid (larger than 7x7 from paper since we use a different network)

In [68]:
# We recommend tuning only the batch size; the remaining hyperparameters should work well
# with the default values. However, you are free to experiment with them.

learning_rate = 1.5e-3
num_epochs = 50
batch_size = 36

# Yolo loss component coefficients (as given in Yolo v1 paper)
lambda_coord = 3
lambda_noobj = 0

## Reading Pascal Data

Since Pascal is a small dataset (5000 in train+val) we have combined the train and val splits to train our detector. This is not typically a good practice, but we will make an exception in this case to be able to get reasonable detection results with a comparatively small object detection dataset.

The train dataset loader also using a variety of data augmentation techniques including random shift, scaling, crop, and flips. Data augmentation is slightly more complicated for detection datasets since the bounding box annotations must be kept consistent throughout the transformations.

Since the output of the detector network we train is an SxSx(B*5+C), we use an encoder to convert the original bounding box coordinates into relative grid bounding box coordinates corresponding to the expected output. We also use a decoder which allows us to convert the opposite direction into image coordinate bounding boxes.

In [69]:
file_root_train = os.path.join(VOC_PATH, 'VOCdevkit_2007/VOC2007/JPEGImages/')
annotation_file_train = 'data/voc2007.txt'

train_dataset = VocDetectorDataset(root_img_dir=file_root_train,dataset_file=annotation_file_train,train=True, S=S)
train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,num_workers=2)
print('Loaded %d train images' % len(train_dataset))

Initializing dataset
Loaded 5011 train images


In [70]:
file_root_test = os.path.join(VOC_PATH, 'VOCdevkit_2007/VOC2007test/JPEGImages/')
annotation_file_test = 'data/voc2007test.txt'

test_dataset = VocDetectorDataset(root_img_dir=file_root_test,dataset_file=annotation_file_test,train=False, S=S)
test_loader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False,num_workers=2)
print('Loaded %d test images' % len(test_dataset))

Initializing dataset
Loaded 4950 test images


In [71]:
data = train_dataset[0]
#print(data)

## Initializing the network

## **Start from here if you modified yolo_loss.py and wish to retrain**

To implement Yolo we will rely on a pretrained classifier as the backbone for our detection network. PyTorch offers a variety of models which are pretrained on ImageNet in the [`torchvision.models`](https://pytorch.org/docs/stable/torchvision/models.html) package. In particular, we will use the ResNet50 architecture as a base for our detector. This is different from the base architecture in the Yolo paper and also results in a different output grid size (14x14 instead of 7x7).

Models are typically pretrained on ImageNet since the dataset is very large (> 1 million images) and widely used. The pretrained model provides a very useful weight initialization for our detector, so that the network is able to learn quickly and effectively.

In [72]:
load_network_path = None
pretrained = True

# use to load a previously trained network
if load_network_path is not None:
    print('Loading saved network from {}'.format(load_network_path))
    net = resnet50().to(device)
    net.load_state_dict(torch.load(load_network_path))
else:
    print('Load pre-trained model')
    net = resnet50(pretrained=pretrained).to(device)

Load pre-trained model


In [73]:
# Print the model
print(net)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

## Set up training tools

In [74]:
from importlib import reload
reload(yolo_loss) # update the import if the implementation has changed

criterion = yolo_loss.YoloLoss(S, B, lambda_coord, lambda_noobj)
optimizer = torch.optim.SGD(net.parameters(), lr=learning_rate, momentum=0.9, weight_decay=5e-4)

## Train detector

In [75]:
best_test_loss = np.inf
learning_rate = learning_rate

os.makedirs('./checkpoints', exist_ok=True)
for epoch in range(num_epochs):
    net.train()

    # Update learning rate late in training
    if epoch == 30 or epoch == 40:
        learning_rate /= 10.0

    for param_group in optimizer.param_groups:
        param_group['lr'] = learning_rate

    print('\n\nStarting epoch %d / %d' % (epoch + 1, num_epochs))
    print('Learning Rate for this epoch: {}'.format(learning_rate))

    total_loss = collections.defaultdict(int)

    for i, data in enumerate(train_loader):
        data = (item.to(device) for item in data)
        images, target_boxes, target_cls, has_object_map = data
        pred = net(images)
        loss_dict = criterion(pred, target_boxes, target_cls, has_object_map)
        for key in loss_dict:
            total_loss[key] += loss_dict[key].item()

        optimizer.zero_grad()
        loss_dict['total_loss'].backward()
        optimizer.step()

        if (i+1) % 50 == 0:
            outstring = 'Epoch [%d/%d], Iter [%d/%d], Loss: ' % ((epoch+1, num_epochs, i+1, len(train_loader)))
            outstring += ', '.join( "%s=%.3f" % (key[:-5], val / (i+1)) for key, val in total_loss.items() )
            print(outstring)

    # evaluate the network on the test data
    if (epoch + 1) % 5 == 0:
        test_aps = evaluate(net, test_dataset_file=annotation_file_test, img_root=file_root_test)
        print(epoch, test_aps)
    with torch.no_grad():
        test_loss = 0.0
        net.eval()
        for i, data in enumerate(test_loader):
            data = (item.to(device) for item in data)
            images, target_boxes, target_cls, has_object_map = data

            pred = net(images)
            loss_dict = criterion(pred, target_boxes, target_cls, has_object_map)
            test_loss += loss_dict['total_loss'].item()
        test_loss /= len(test_loader)

    if best_test_loss > test_loss:
        best_test_loss = test_loss
        print('Updating best test loss: %.5f' % best_test_loss)
        torch.save(net.state_dict(),'checkpoints/best_detector.pth')

    if (epoch+1) in [5, 10, 20, 30, 40]:
        torch.save(net.state_dict(),'checkpoints/detector_epoch_%d.pth' % (epoch+1))

    torch.save(net.state_dict(),'checkpoints/detector.pth')





Starting epoch 1 / 50
Learning Rate for this epoch: 0.0015
Epoch [1/50], Iter [50/140], Loss: total=10.097, reg=1.329, containing_obj=1.888, no_obj=0.000, cls=6.880
Epoch [1/50], Iter [100/140], Loss: total=9.633, reg=1.214, containing_obj=1.785, no_obj=0.000, cls=6.634
Updating best test loss: 9.19107


Starting epoch 2 / 50
Learning Rate for this epoch: 0.0015
Epoch [2/50], Iter [50/140], Loss: total=8.742, reg=0.915, containing_obj=1.611, no_obj=0.000, cls=6.216
Epoch [2/50], Iter [100/140], Loss: total=8.633, reg=0.881, containing_obj=1.596, no_obj=0.000, cls=6.155
Updating best test loss: 8.66166


Starting epoch 3 / 50
Learning Rate for this epoch: 0.0015
Epoch [3/50], Iter [50/140], Loss: total=8.017, reg=0.756, containing_obj=1.488, no_obj=0.000, cls=5.773
Epoch [3/50], Iter [100/140], Loss: total=8.102, reg=0.751, containing_obj=1.506, no_obj=0.000, cls=5.845
Updating best test loss: 8.35866


Starting epoch 4 / 50
Learning Rate for this epoch: 0.0015
Epoch [4/50], Iter [50/

100%|██████████| 4950/4950 [07:02<00:00, 11.72it/s]


---class aeroplane ap 0.09617334532759454---
---class bicycle ap 0.07377932439327411---
---class bird ap 0.07804977818472557---
---class boat ap 0.011118592034159679---
---class bottle ap 0.0020488837086852485---
---class bus ap 0.07010340914151213---
---class car ap 0.17544776705881818---
---class cat ap 0.3246936575256085---
---class chair ap 0.005188171766603446---
---class cow ap 0.037939539752933724---
---class diningtable ap 0.008617147644246198---
---class dog ap 0.24113130468417326---
---class horse ap 0.24174312285974703---
---class motorbike ap 0.11669055987374985---
---class person ap 0.15447442612465162---
---class pottedplant ap 0.0011613424261851634---
---class sheep ap 0.010140956256239345---
---class sofa ap 0.008238308875557626---
---class train ap 0.2083506349973397---
---class tvmonitor ap 0.008496658330349603---
---map 0.09367934654830773---
4 [np.float64(0.09617334532759454), np.float64(0.07377932439327411), np.float64(0.07804977818472557), np.float64(0.01111859203

100%|██████████| 4950/4950 [07:02<00:00, 11.71it/s]


---class aeroplane ap 0.2540157931489565---
---class bicycle ap 0.26855540607302714---
---class bird ap 0.21448697716951373---
---class boat ap 0.051956740410814745---
---class bottle ap 0.008003283765036541---
---class bus ap 0.34181914949363923---
---class car ap 0.2881487174352909---
---class cat ap 0.45138280701727834---
---class chair ap 0.0643352120054229---
---class cow ap 0.1159092305250426---
---class diningtable ap 0.1226791930456341---
---class dog ap 0.36239507470509463---
---class horse ap 0.30401325228889153---
---class motorbike ap 0.3070342011002265---
---class person ap 0.2197497424075425---
---class pottedplant ap 0.006441404221719046---
---class sheep ap 0.047464552007211495---
---class sofa ap 0.22539717752813437---
---class train ap 0.3373883565216361---
---class tvmonitor ap 0.07641674566729284---
---map 0.2033796508268703---
9 [np.float64(0.2540157931489565), np.float64(0.26855540607302714), np.float64(0.21448697716951373), np.float64(0.051956740410814745), np.fl

100%|██████████| 4950/4950 [07:00<00:00, 11.77it/s]


---class aeroplane ap 0.3308998460571204---
---class bicycle ap 0.3755386487013267---
---class bird ap 0.2984442213076891---
---class boat ap 0.14412829711465353---
---class bottle ap 0.020928213997768206---
---class bus ap 0.4082182834263734---
---class car ap 0.3623613804340508---
---class cat ap 0.5124130364606891---
---class chair ap 0.09259451044356805---
---class cow ap 0.1514355099282795---
---class diningtable ap 0.1965786091384955---
---class dog ap 0.4353928044827831---
---class horse ap 0.41969689433300833---
---class motorbike ap 0.4189200308418799---
---class person ap 0.26754722376117535---
---class pottedplant ap 0.029174352800825674---
---class sheep ap 0.14103974661784963---
---class sofa ap 0.229415070846---
---class train ap 0.4617371891591479---
---class tvmonitor ap 0.16391386517083167---
---map 0.2730188867511758---
14 [np.float64(0.3308998460571204), np.float64(0.3755386487013267), np.float64(0.2984442213076891), np.float64(0.14412829711465353), np.float64(0.0209

100%|██████████| 4950/4950 [07:01<00:00, 11.76it/s]


---class aeroplane ap 0.3600029555782397---
---class bicycle ap 0.4642761139578383---
---class bird ap 0.33512292708378466---
---class boat ap 0.1585915826956625---
---class bottle ap 0.031063970228294043---
---class bus ap 0.4589523890356745---
---class car ap 0.4276767001624094---
---class cat ap 0.5658938512059805---
---class chair ap 0.12891072843475512---
---class cow ap 0.28036104841177034---
---class diningtable ap 0.21830611167686362---
---class dog ap 0.507266923968333---
---class horse ap 0.4704639300782968---
---class motorbike ap 0.4416653106722957---
---class person ap 0.32826684852433985---
---class pottedplant ap 0.06248546294897308---
---class sheep ap 0.2323142860869261---
---class sofa ap 0.25023581114464005---
---class train ap 0.5253372925743164---
---class tvmonitor ap 0.2370720983169956---
---map 0.32421331713931945---
19 [np.float64(0.3600029555782397), np.float64(0.4642761139578383), np.float64(0.33512292708378466), np.float64(0.1585915826956625), np.float64(0.0

KeyboardInterrupt: 

# View example predictions

In [ ]:
net.eval()

# select random image from test set
image_name = random.choice(test_dataset.fnames)
image = cv2.imread(os.path.join(file_root_test, image_name))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print('predicting...')
result = predict_image(net, image_name, root_img_directory=file_root_test)
for left_up, right_bottom, class_name, _, prob in result:
    color = COLORS[VOC_CLASSES.index(class_name)]
    cv2.rectangle(image, left_up, right_bottom, color, 2)
    label = class_name + str(round(prob, 2))
    text_size, baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
    p1 = (left_up[0], left_up[1] - text_size[1])
    cv2.rectangle(image, (p1[0] - 2 // 2, p1[1] - 2 - baseline), (p1[0] + text_size[0], p1[1] + text_size[1]),
                  color, -1)
    cv2.putText(image, label, (p1[0], p1[1] + baseline), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, 8)

plt.figure(figsize = (15,15))
plt.imshow(image)


## Evaluate on Test

To evaluate detection results we use mAP (mean of average precision over each class)

In [ ]:
test_aps = evaluate(net, test_dataset_file=annotation_file_test, img_root=file_root_test)

### Cell added to get intermediate mAP values for students

In [ ]:
network_paths = ['./checkpoints/detector_epoch_%d.pth' % epoch for epoch in [5, 10, 20, 30, 40]]+['./checkpoints/detector.pth']
for load_network_path in network_paths:
    print('Loading saved network from {}'.format(load_network_path))
    net_loaded =  resnet50().to(device)
    net_loaded.load_state_dict(torch.load(load_network_path))
    evaluate(net_loaded, test_dataset_file=annotation_file_test, img_root=file_root_test)


In [ ]:
output_submission_csv('my_solution.csv', test_aps)